In [29]:
%load_ext autoreload
%autoreload 2

In [30]:
from pathlib import Path
import sys

PROJECT_ROOT = Path().resolve().parents[0]  # notebooks/ is one level down
sys.path.append(str(PROJECT_ROOT / "src"))

In [31]:
from chatGnT.config import CFG, ensure_dirs
import chatGnT.utils as utils
import chatGnT.data.load as load
import chatGnT.data.preprocess as preprocess
import chatGnT.data.tokenize as tokenize
ensure_dirs(CFG)

In [49]:
import time
import math
import torch
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn  # for embedding layer
import chatGnT.models.transformer as transformer
import chatGnT.models.positional_encoding as positional_encoding

In [33]:
data = load.load_all()
df = data["ingred"]


In [34]:
#TODO: revisit and handle things like "Twist of  Orange peel" "Top with..."
#TODO: how make units consistent - good opportunity for test functions?
df_clean = preprocess.clean_ingred(df)
# df_clean.head()


In [35]:
tokens = tokenize.recipe_to_tokens(df_clean)
print(tokens[0])

['<amt>1</amt>', '<unit>oz</unit>', '<ingred>Coconut rum</ingred>', '<sep>', '<amt>1/2</amt>', '<unit>oz</unit>', '<ingred>Amaretto</ingred>', '<sep>', '<amt>4</amt>', '<unit>oz</unit>', '<ingred>Orange juice</ingred>', '<sep>', '<amt>1/2</amt>', '<unit>oz</unit>', '<ingred>Grenadine</ingred>', '<sep>']


In [36]:
vocab = tokenize.make_vocab(tokens)
inv_vocab = tokenize.invert_vocab(vocab)

tokens_padded = tokenize.embed_tokens(tokens, vocab)

In [37]:
#TODO: this just mapping tokens to vocab, not sure how checked recipe...
decoded_recipe = tokenize.invert_vocab(vocab)
# print("Decoded back:", decoded_recipe)

In [38]:
# tokens_tensor = [torch.tensor(r) for r in tokens_padded]
tokens_tensor = torch.tensor(tokens_padded, dtype=torch.long)
print(tokens_tensor.shape)  # torch.Size([621, 48])

# Need to have [seq_len, batch_size] for TransformerEncoder
tokens_tensor = tokens_tensor.transpose(0, 1)
print(tokens_tensor.shape)

torch.Size([621, 49])
torch.Size([49, 621])


In [39]:
# TODO: for now, add padding mask but not a causal mask that block seeing future
pad_id = vocab["<pad>"]   # should be 0
# src_key_padding_mask shape = [batch_size, seq_len]
pad_mask = (tokens_tensor == pad_id).transpose(0, 1)
print(pad_mask.shape)   # [batch_size, seq_len]


torch.Size([621, 49])


In [42]:
embed_size=16
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = transformer.TransformerModel(
    ntoken=len(vocab),
    ninp=embed_size,
    nhead=4,
    nhid=256,
    nlayers=2).to(device)
#TODO: investigate error? 


/Users/slacksa/repos/chatGnT/src/chatGnT/models/transformer.py:16: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.transformer_encoder = TransformerEncoder(encoder_layers, nlayers)


In [43]:
learning_rate = 1e-3
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = torch.nn.CrossEntropyLoss(ignore_index=pad_id)

In [50]:
def train():
    model.train() # Turn on the train mode
    total_loss = 0.
    start_time = time.time()
    
    seq_len = train_src.size(0)
    src_mask = model.generate_square_subsequent_mask(seq_len).to(device)
    # src_mask = model.generate_square_subsequent_mask(bptt).to(device)
    # for batch, i in enumerate(range(0, train_data.size(0) - 1, bptt)):
    # data, targets = get_batch(train_data, i)
    optimizer.zero_grad()
    # if data.size(0) != bptt:
    #     src_mask = model.generate_square_subsequent_mask(data.size(0)).to(device)

    # Forward pass
    output = model(
        src=train_src,
        src_key_padding_mask=train_pad_mask[:, :-1],
        src_mask=src_mask)

    loss = criterion(
        output.view(-1, ntokens),
        train_tgt.reshape(-1))

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
    optimizer.step()

    total_loss += loss.item()
    # log_interval = 200
    # if batch % log_interval == 0 and batch > 0:
        # cur_loss = total_loss / log_interval
    elapsed = time.time() - start_time
    print('| epoch {:3d} | '
            'lr {:.6f} | '
            'loss {:5.2f} | ppl {:8.2f}'.format(
            epoch,
            optimizer.param_groups[0]['lr'],
            elapsed,
            loss.item(),
            math.exp(loss.item())))
    # total_loss = 0
    start_time = time.time()

In [51]:
def evaluate(eval_model):
    eval_model.eval() # Turn on the evaluation mode
    # total_loss = 0.

    seq_len = train_src.size(0)
    src_mask = model.generate_square_subsequent_mask(seq_len).to(device)

    with torch.no_grad():
        # for i in range(0, data_source.size(0) - 1, bptt):
            # data, targets = get_batch(data_source, i)
            # if data.size(0) != bptt:
                # src_mask = model.generate_square_subsequent_mask(data.size(0)).to(device)
            output = model(
                src=train_src,
                src_key_padding_mask=train_pad_mask[:, :-1],
                src_mask=src_mask)
            output_flat = output.view(-1, ntokens)
            # total_loss += len(data) * criterion(output_flat, targets).item()
            loss = criterion(
                output.reshape(-1, ntokens),
                val_tgt.reshape(-1))
    # return total_loss / (len(data_source) - 1)
    return loss.item()

In [52]:
best_val_loss = float("inf")
epochs = 10 # The number of epochs
best_model = None
ntokens = len(vocab)

# For training
train_src = tokens_tensor[:-1, :]       # all tokens except last
train_tgt = tokens_tensor[1:, :]        # shifted by one
train_pad_mask = (tokens_tensor == pad_id).transpose(0, 1)

# For validation, if you have a separate val_tokens_tensor
#TODO: eventually update so not same as training
val_src = tokens_tensor[:-1, :]
val_tgt = tokens_tensor[1:, :]
val_pad_mask = (tokens_tensor == pad_id).transpose(0, 1)

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()

    train_loss = train()
    val_loss = evaluate(model)

    print('-' * 89)
    print('| end of epoch {:3d} | time: {:5.2f}s | '
          'valid loss {:5.2f} | valid ppl {:8.2f}'.format(
              epoch,
              (time.time() - epoch_start_time),
              val_loss,
              math.exp(val_loss)))
    print('-' * 89)


PositionalEncoding forward called, input shape: torch.Size([48, 621, 16])
| epoch   1 | lr 0.001000 | loss  0.25 | ppl     6.07
PositionalEncoding forward called, input shape: torch.Size([48, 621, 16])
-----------------------------------------------------------------------------------------
| end of epoch   1 | time:  0.28s | valid loss  5.96 | valid ppl   388.06
-----------------------------------------------------------------------------------------
PositionalEncoding forward called, input shape: torch.Size([48, 621, 16])
| epoch   2 | lr 0.001000 | loss  0.24 | ppl     6.05
PositionalEncoding forward called, input shape: torch.Size([48, 621, 16])
-----------------------------------------------------------------------------------------
| end of epoch   2 | time:  0.27s | valid loss  5.93 | valid ppl   377.68
-----------------------------------------------------------------------------------------
PositionalEncoding forward called, input shape: torch.Size([48, 621, 16])
| epoch   3 | 